# Model Communication Layer — Interactive Walkthrough

This notebook re-implements the **core logic** of Easy Agent's Model Communication Layer
(`src/services/api/`, `src/services/mcp/`, `src/services/skills/`, `src/types/`, `src/utils/`)
as standalone Python. Every class, function, and constant is inlined — **no project imports**.

**What you'll learn:**
| # | Subsystem | Key Concepts |
|---|-----------|-------------|
| 1 | Type Foundations | ContentBlock, StreamEvent, McpServerConnection, Skill |
| 2 | API Client | Lazy singleton, token limit constants |
| 3 | Streaming Engine | AsyncGenerator, per-index JSON accumulation, escalated retry |
| 4 | Token Estimation | Character-based heuristic, hybrid usage+estimation, budget snapshots |
| 5 | MCP Name Utilities | Normalization, `mcp__server__tool` convention |
| 6 | MCP Registry | In-memory connection+tool store |
| 7 | MCP Config Loading | Two-scope settings, transport validation |
| 8 | MCP Client | Transport factory, 30s timeout, SIGINT→SIGTERM→SIGKILL cleanup |
| 9 | MCP Tool Discovery | `tools/list` → Tool adapter with `call()` forwarding |
| 10 | Skills Frontmatter | YAML split, field normalization, fallback description |
| 11 | Skills Registry | Dual-map (dynamic / conditional), one-way activation |
| 12 | Skills Budget | Three-tier degradation, system-reminder injection |
| 13 | Stream Debug | Opt-in JSONL logging |

> Source: `easy-agent-main/src/` — TypeScript → Python transliteration preserving logic fidelity.

## 0. Setup — Imports & Path Discovery

In [ ]:
from pathlib import Path
import os, sys, json, re, math, asyncio, time, signal, atexit
from dataclasses import dataclass, field
from typing import Union, Optional, Callable, Awaitable, Any
from datetime import datetime

# Path discovery — works regardless of where you launch the notebook
_cwd = Path(".").resolve()
PROJECT_ROOT = _cwd
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "pyproject.toml").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    # Fallback: use the known location
    PROJECT_ROOT = Path("/Users/nick/Code/ds_workstation/easy-agent-main")

print(f"Project root: {PROJECT_ROOT}")

---
## 1. Type Foundations — Message & Content Block Types

These dataclasses mirror `src/types/message.ts`. Every piece of content exchanged
between Easy Agent and the LLM is expressed through these types.

**Key design decisions from the source:**
- `ThinkingBlock` is preserved (with `signature`) so it can be echoed back on the next turn.
- `ToolResultBlock.content` accepts `str | list[ContentBlock]` (dual-format for MCP image results).
- `Usage` tracks optional `cache_creation_input_tokens` / `cache_read_input_tokens`.

In [ ]:
@dataclass
class TextBlock:
    type: str = "text"
    text: str = ""

@dataclass
class ToolUseBlock:
    type: str = "tool_use"
    id: str = ""
    name: str = ""
    input: dict = field(default_factory=dict)

@dataclass
class ToolResultBlock:
    type: str = "tool_result"
    tool_use_id: str = ""
    content: Union[str, list] = ""
    is_error: bool = False

@dataclass
class ThinkingBlock:
    """Extended-thinking block — preserved for echo on next turn."""
    type: str = "thinking"
    thinking: str = ""
    signature: Optional[str] = None

ContentBlock = Union[TextBlock, ToolUseBlock, ToolResultBlock, ThinkingBlock]

@dataclass
class UserMessage:
    role: str = "user"
    content: Union[str, list] = ""

@dataclass
class AssistantMessage:
    role: str = "assistant"
    content: Union[str, list] = ""

Message = Union[UserMessage, AssistantMessage]

@dataclass
class Usage:
    input_tokens: int = 0
    output_tokens: int = 0
    cache_creation_input_tokens: Optional[int] = None
    cache_read_input_tokens: Optional[int] = None

# Quick smoke test
u = Usage(input_tokens=100, output_tokens=50, cache_creation_input_tokens=200)
assert u.input_tokens == 100
assert u.cache_creation_input_tokens == 200
print("Content block types OK")
print(f"  TextBlock fields:    {[f.name for f in TextBlock.__dataclass_fields__.values()]}")
print(f"  ToolUseBlock fields: {[f.name for f in ToolUseBlock.__dataclass_fields__.values()]}")
print(f"  Usage fields:        {[f.name for f in Usage.__dataclass_fields__.values()]}")

### 1.2 Stream Event Discriminated Union

The streaming system yields these events incrementally. The agentic loop and UI
consume them as they arrive.

In [ ]:
@dataclass
class StreamTextEvent:
    type: str = "text"
    text: str = ""

@dataclass
class StreamToolUseStartEvent:
    type: str = "tool_use_start"
    id: str = ""
    name: str = ""

@dataclass
class StreamToolUseInputEvent:
    type: str = "tool_use_input"
    id: str = ""
    partial_json: str = ""

@dataclass
class StreamMessageStartEvent:
    type: str = "message_start"
    message_id: str = ""

@dataclass
class StreamMessageDoneEvent:
    type: str = "message_done"
    stop_reason: str = ""
    usage: Optional[Usage] = None

@dataclass
class StreamErrorEvent:
    type: str = "error"
    error: Optional[Exception] = None

StreamEvent = Union[
    StreamTextEvent, StreamToolUseStartEvent, StreamToolUseInputEvent,
    StreamMessageStartEvent, StreamMessageDoneEvent, StreamErrorEvent,
]

print("Stream event types OK — 6 event kinds defined")

### 1.3 MCP Connection State Types (`src/types/mcp.ts`)

Models the full lifecycle: Pending → Connected | Failed | Disabled.

In [ ]:
@dataclass
class McpStdioServerConfig:
    type: str = "stdio"
    command: str = ""
    args: list = field(default_factory=list)
    env: dict = field(default_factory=dict)

@dataclass
class McpHTTPServerConfig:
    type: str = "http"
    url: str = ""
    headers: dict = field(default_factory=dict)

@dataclass
class McpSSEServerConfig:
    type: str = "sse"
    url: str = ""
    headers: dict = field(default_factory=dict)

McpServerConfig = Union[McpStdioServerConfig, McpHTTPServerConfig, McpSSEServerConfig]

@dataclass
class ScopedMcpServerConfig:
    """Server config tagged with its origin scope (user / project)."""
    scope: str = ""
    # Transport fields are set dynamically based on type
    type: str = "stdio"
    command: str = ""
    args: list = field(default_factory=list)
    env: dict = field(default_factory=dict)
    url: str = ""
    headers: dict = field(default_factory=dict)

@dataclass
class ConnectedMcpServer:
    name: str = ""
    type: str = "connected"
    client: Any = None           # MCP SDK Client instance
    capabilities: Any = None     # ServerCapabilities
    server_info: Optional[dict] = None
    config: Optional[ScopedMcpServerConfig] = None
    cleanup: Optional[Callable] = None

@dataclass
class FailedMcpServer:
    name: str = ""
    type: str = "failed"
    config: Optional[ScopedMcpServerConfig] = None
    error: str = ""

@dataclass
class PendingMcpServer:
    name: str = ""
    type: str = "pending"
    config: Optional[ScopedMcpServerConfig] = None
    started_at: int = 0

McpServerConnection = Union[ConnectedMcpServer, FailedMcpServer, PendingMcpServer]

# Smoke test
p = PendingMcpServer(name="github", type="pending", started_at=int(time.time() * 1000))
assert p.type == "pending"
print(f"MCP types OK — PendingMcpServer(name={p.name}, started_at={p.started_at})")

### 1.4 Skill Type System (`src/types/types.ts`)

Skills are declarative prompt templates: Markdown files with YAML frontmatter.

In [ ]:
@dataclass
class SkillFrontmatter:
    name: Optional[str] = None
    description: Optional[str] = None
    when_to_use: Optional[str] = None
    allowed_tools: list = field(default_factory=list)
    argument_hint: Optional[str] = None
    disable_model_invocation: bool = False
    paths: Optional[list] = None     # gitignore-style conditional activation
    has_fork_context: bool = False
    raw: dict = field(default_factory=dict)  # untouched YAML for forward compat

@dataclass
class Skill:
    name: str = ""
    description: str = ""
    when_to_use: Optional[str] = None
    body: str = ""                   # Markdown without frontmatter
    file_path: str = ""              # Absolute, realpath-resolved
    base_dir: str = ""               # Directory containing SKILL.md
    source: str = ""                 # "user" or "project"
    frontmatter: Optional[SkillFrontmatter] = None

# Smoke test
fm = SkillFrontmatter(paths=["**/*.test.ts"], allowed_tools=["Read", "Glob"])
sk = Skill(name="test-reviewer", description="Reviews test files", frontmatter=fm)
assert sk.frontmatter.paths == ["**/*.test.ts"]
print(f"Skill types OK — Skill(name={sk.name}, conditional={sk.frontmatter.paths is not None})")

---
## 2. API Client Subsystem (`src/services/api/client.ts`)

A thin wrapper around the Anthropic SDK. Manages a lazily-initialized singleton
client instance plus token limit constants.

**Constants from the source:**
| Constant | Value | Purpose |
|----------|-------|---------|
| `CAPPED_DEFAULT_MAX_TOKENS` | 8,000 | Default output cap |
| `ESCALATED_MAX_TOKENS` | 64,000 | Retry cap on truncation |
| `COMPACT_MAX_OUTPUT_TOKENS` | 20,000 | Compaction calls |
| `MAX_OUTPUT_TOKENS_RECOVERY_LIMIT` | 3 | Max continuation attempts |

In [ ]:
DEFAULT_MODEL = os.environ.get("ANTHROPIC_MODEL", "claude-sonnet-4-20250514")
CAPPED_DEFAULT_MAX_TOKENS = 8_000
ESCALATED_MAX_TOKENS = 64_000
COMPACT_MAX_OUTPUT_TOKENS = 20_000
MAX_OUTPUT_TOKENS_RECOVERY_LIMIT = 3
DEFAULT_MAX_TOKENS = CAPPED_DEFAULT_MAX_TOKENS

_client_instance = None

def get_anthropic_client(api_key=None, base_url=None):
    """
    Get or create the Anthropic client (lazy singleton).

    The SDK reads ANTHROPIC_AUTH_TOKEN from env. Pass api_key to override.
    When no options and a cached instance exists, returns the cache (fast path).
    """
    global _client_instance
    if _client_instance is not None and api_key is None and base_url is None:
        return _client_instance

    # In the real codebase this creates an Anthropic() SDK instance.
    # We simulate with a dict for notebook portability.
    client = {
        "api_key": api_key or os.environ.get("ANTHROPIC_AUTH_TOKEN", ""),
        "base_url": base_url or os.environ.get("ANTHROPIC_BASE_URL", ""),
        "model": DEFAULT_MODEL,
    }
    if api_key is None and base_url is None:
        _client_instance = client
    return client

def reset_client():
    """Drop the cached instance. Called when API key changes at runtime."""
    global _client_instance
    _client_instance = None

# Demo
c = get_anthropic_client()
print(f"Client created: model={c['model']}, base_url={c['base_url'][:40] or '(default)'}")
print(f"Singleton check: get_anthropic_client() is get_anthropic_client() = {get_anthropic_client() is c}")
reset_client()
print(f"After reset: cached = {_client_instance}")

---
## 3. Streaming Engine (`src/services/api/streaming.ts`)

The most critical file — **every LLM interaction** flows through `streamMessage()`.

**Key design:**
- `AsyncGenerator` that yields `StreamEvent` objects and returns a `StreamResult`
- **Per-index JSON accumulation** for tool input (prevents corruption with overlapping blocks)
- Errors are **yielded** as events, not thrown (graceful handling in the generator protocol)

> The actual streaming uses the Anthropic SDK's `client.messages.stream()`.
> Here we simulate the event stream to demonstrate the accumulation logic.

In [ ]:
@dataclass
class StreamResult:
    assistant_message: AssistantMessage
    usage: Usage
    stop_reason: str

def simulate_stream_events():
    """
    Simulate the SSE events the Anthropic API would emit for:
      'Hello! I'll use the Read tool.'  +  tool_use Read(file_path='foo.py')

    This demonstrates the exact event sequence that streamMessage() processes.
    """
    return [
        {"type": "message_start", "message": {"id": "msg_01ABC", "usage": {"input_tokens": 150, "output_tokens": 0}}},
        {"type": "content_block_start", "index": 0, "content_block": {"type": "text", "text": ""}},
        {"type": "content_block_delta", "index": 0, "delta": {"type": "text_delta", "text": "Hello!"}},
        {"type": "content_block_delta", "index": 0, "delta": {"type": "text_delta", "text": " I'll"}},
        {"type": "content_block_delta", "index": 0, "delta": {"type": "text_delta", "text": " use the Read tool."}},
        {"type": "content_block_stop", "index": 0},
        {"type": "content_block_start", "index": 1, "content_block": {"type": "tool_use", "id": "tu_01", "name": "Read", "input": {}}},
        {"type": "content_block_delta", "index": 1, "delta": {"type": "input_json_delta", "partial_json": '{"file_'}},
        {"type": "content_block_delta", "index": 1, "delta": {"type": "input_json_delta", "partial_json": 'path": "foo.py"}'}},
        {"type": "content_block_stop", "index": 1},
        {"type": "message_delta", "delta": {"stop_reason": "tool_use"}, "usage": {"output_tokens": 25}},
        {"type": "message_stop"},
    ]

def stream_message_simulated(events):
    """
    Python port of streamMessage() from streaming.ts.

    Demonstrates the core pattern:
    1. Maintain state accumulators (contentBlocks, toolInputJsonByIndex, usage)
    2. Switch on event.type
    3. Yield StreamEvents incrementally
    4. Return StreamResult when done
    """
    content_blocks = []           # ContentBlock[] — indexed by block position
    tool_input_json_by_index = {} # Map[int, str] — per-block JSON accumulation
    message_id = ""
    stop_reason = ""
    usage = Usage()

    for event in events:
        etype = event["type"]

        if etype == "message_start":
            message_id = event["message"]["id"]
            u = event["message"].get("usage", {})
            usage.input_tokens = u.get("input_tokens", 0)
            usage.output_tokens = u.get("output_tokens", 0)
            yield StreamMessageStartEvent(message_id=message_id)

        elif etype == "content_block_start":
            index = event["index"]
            cb = event["content_block"]
            if cb["type"] == "text":
                content_blocks.append(TextBlock(text=""))
            elif cb["type"] == "tool_use":
                content_blocks.append(ToolUseBlock(id=cb["id"], name=cb["name"], input={}))
                tool_input_json_by_index[index] = ""
                yield StreamToolUseStartEvent(id=cb["id"], name=cb["name"])

        elif etype == "content_block_delta":
            delta = event["delta"]
            index = event["index"]

            if delta.get("type") == "text_delta":
                content_blocks[index].text += delta["text"]
                yield StreamTextEvent(text=delta["text"])

            elif delta.get("type") == "input_json_delta":
                # KEY DETAIL: accumulate per-index, not in a shared buffer
                prev = tool_input_json_by_index.get(index, "")
                tool_input_json_by_index[index] = prev + delta["partial_json"]
                block = content_blocks[index]
                if isinstance(block, ToolUseBlock):
                    yield StreamToolUseInputEvent(id=block.id, partial_json=delta["partial_json"])

        elif etype == "content_block_stop":
            index = event["index"]
            block = content_blocks[index]
            accumulated = tool_input_json_by_index.get(index)
            if isinstance(block, ToolUseBlock) and accumulated:
                try:
                    block.input = json.loads(accumulated)
                except json.JSONDecodeError:
                    block.input = {"_raw": accumulated}
            tool_input_json_by_index.pop(index, None)

        elif etype == "message_delta":
            du = event.get("usage", {})
            usage.output_tokens = du.get("output_tokens", usage.output_tokens)
            stop_reason = event["delta"].get("stop_reason", "")

        elif etype == "message_stop":
            yield StreamMessageDoneEvent(stop_reason=stop_reason, usage=usage)

    return StreamResult(
        assistant_message=AssistantMessage(role="assistant", content=[b for b in content_blocks if b]),
        usage=usage,
        stop_reason=stop_reason,
    )

# ── Run the simulation ──
# In Python, generators use `return value` → StopIteration.value.
# We must manually iterate (not for-loop) to capture the return value.
events = simulate_stream_events()
gen = stream_message_simulated(events)
print("=== Streaming Simulation ===\n")
result = None
while True:
    try:
        event = next(gen)
        if isinstance(event, StreamMessageStartEvent):
            print(f"[message_start] id={event.message_id}")
        elif isinstance(event, StreamTextEvent):
            print(f"[text] {repr(event.text)}")
        elif isinstance(event, StreamToolUseStartEvent):
            print(f"[tool_use_start] id={event.id}, name={event.name}")
        elif isinstance(event, StreamToolUseInputEvent):
            print(f"[tool_use_input] id={event.id}, json={repr(event.partial_json)}")
        elif isinstance(event, StreamMessageDoneEvent):
            print(f"[message_done] stop={event.stop_reason}, usage={event.usage}")
    except StopIteration as e:
        result = e.value
        break

print(f"\n=== Final StreamResult ===")
print(f"  stop_reason: {result.stop_reason}")
print(f"  usage: input={result.usage.input_tokens}, output={result.usage.output_tokens}")
for i, block in enumerate(result.assistant_message.content):
    if isinstance(block, TextBlock):
        print(f"  block[{i}]: TextBlock(text={repr(block.text)})")
    elif isinstance(block, ToolUseBlock):
        print(f"  block[{i}]: ToolUseBlock(name={block.name}, input={block.input})")

### 3.1 Per-Index JSON Accumulation — Why It Matters

Some providers (MiniMax, certain Anthropic-compatible shims) emit overlapping content
blocks — starting `content_block_start` for block 1 before `content_block_stop` of block 0.
A shared buffer would corrupt the JSON. Per-index buffers prevent this.

In [ ]:
def demonstrate_per_index_accumulation():
    """Show the difference between shared vs per-index buffer strategies."""

    # Simulated overlapping events (provider bug)
    overlapping_events = [
        {"type": "content_block_start", "index": 0, "content_block": {"type": "tool_use", "id": "tu_0", "name": "Read", "input": {}}},
        {"type": "content_block_delta", "index": 0, "delta": {"type": "input_json_delta", "partial_json": '{"file_path": "a.py"}'}},
        # Block 1 starts BEFORE block 0 stops (overlapping!)
        {"type": "content_block_start", "index": 1, "content_block": {"type": "tool_use", "id": "tu_1", "name": "Write", "input": {}}},
        {"type": "content_block_delta", "index": 1, "delta": {"type": "input_json_delta", "partial_json": '{"file_path": "b.py"}'}},
        {"type": "content_block_stop", "index": 0},
        {"type": "content_block_stop", "index": 1},
    ]

    # ── BROKEN: Shared buffer ──
    shared_buf = ""
    shared_results = {}
    for ev in overlapping_events:
        if ev["type"] == "content_block_delta" and ev["delta"]["type"] == "input_json_delta":
            shared_buf += ev["delta"]["partial_json"]
        elif ev["type"] == "content_block_stop":
            shared_results[ev["index"]] = shared_buf
            shared_buf = ""

    print("=== Shared Buffer (BROKEN) ===")
    for idx, val in shared_results.items():
        try:
            parsed = json.loads(val)
            print(f"  block[{idx}]: {parsed}")
        except json.JSONDecodeError:
            print(f"  block[{idx}]: PARSE FAILED → {repr(val)}")

    # ── CORRECT: Per-index buffers ──
    per_index = {}
    for ev in overlapping_events:
        if ev["type"] == "content_block_delta" and ev["delta"]["type"] == "input_json_delta":
            idx = ev["index"]
            per_index[idx] = per_index.get(idx, "") + ev["delta"]["partial_json"]
        elif ev["type"] == "content_block_stop":
            pass  # would parse per_index[ev["index"]] here

    print("\n=== Per-Index Buffers (CORRECT) ===")
    for idx in sorted(per_index):
        parsed = json.loads(per_index[idx])
        print(f"  block[{idx}]: {parsed}")

demonstrate_per_index_accumulation()

### 3.2 Escalated Retry (`streamMessageWithRetry`)

When the model hits `max_tokens` (truncated output), automatically retries at 64K tokens.

In [ ]:
def stream_message_with_retry_simulated(params_max_tokens=8000):
    """
    Demonstrates the escalation logic from streamMessageWithRetry():

    1. Try at params.max_tokens (default 8K)
    2. If stopReason == 'max_tokens', retry at ESCALATED_MAX_TOKENS (64K)
    3. If still truncated, return what we have (caller does multi-turn recovery)
    """
    # Simulate: first attempt truncates, second succeeds
    first_events = [
        {"type": "message_start", "message": {"id": "msg_try1", "usage": {"input_tokens": 100}}},
        {"type": "content_block_start", "index": 0, "content_block": {"type": "text", "text": ""}},
        {"type": "content_block_delta", "index": 0, "delta": {"type": "text_delta", "text": "Long output..." * 100}},
        {"type": "content_block_stop", "index": 0},
        {"type": "message_delta", "delta": {"stop_reason": "max_tokens"}, "usage": {"output_tokens": 8000}},
        {"type": "message_stop"},
    ]

    second_events = [
        {"type": "message_start", "message": {"id": "msg_try2", "usage": {"input_tokens": 100}}},
        {"type": "content_block_start", "index": 0, "content_block": {"type": "text", "text": ""}},
        {"type": "content_block_delta", "index": 0, "delta": {"type": "text_delta", "text": "Complete output."}},
        {"type": "content_block_stop", "index": 0},
        {"type": "message_delta", "delta": {"stop_reason": "end_turn"}, "usage": {"output_tokens": 50}},
        {"type": "message_stop"},
    ]

    print("Attempt 1: max_tokens=8000")
    gen1 = stream_message_simulated(first_events)
    r1 = None
    while True:
        try:
            next(gen1)
        except StopIteration as e:
            r1 = e.value
            break
    print(f"  stop_reason={r1.stop_reason}, output_tokens={r1.usage.output_tokens}")

    if r1.stop_reason == "max_tokens":
        print(f"\nTruncated! Escalating to {ESCALATED_MAX_TOKENS} tokens...")
        print("Attempt 2: max_tokens=64000")
        gen2 = stream_message_simulated(second_events)
        r2 = None
        while True:
            try:
                next(gen2)
            except StopIteration as e:
                r2 = e.value
                break
        print(f"  stop_reason={r2.stop_reason}, output_tokens={r2.usage.output_tokens}")
        return r2
    return r1

result = stream_message_with_retry_simulated()
print(f"\nFinal: stop_reason={result.stop_reason}")

---
## 4. Token Estimation & Budget Management (`src/utils/tokens.ts`)

Character-based heuristic (no native tokenizer dependency). Used by compaction
and the agentic loop to prevent context overflow.

**Heuristic ratios:**
| Content | Chars/Token | Rationale |
|---------|-------------|-----------|
| Plain text | 4 | Average English |
| JSON | 2 | More token-dense |
| Binary (images) | Fixed 2,000 | Conservative |
| Message overhead | 12 tokens | Role + framing |
| Tool block overhead | 24 tokens | Name + schema |

In [ ]:
MODEL_CONTEXT_WINDOW_DEFAULT = 200_000
MAX_OUTPUT_TOKENS_FOR_SUMMARY = 20_000
AUTOCOMPACT_BUFFER_TOKENS = 13_000
MANUAL_COMPACT_BUFFER_TOKENS = 3_000

TEXT_CHARS_PER_TOKEN = 4
JSON_CHARS_PER_TOKEN = 2
MESSAGE_OVERHEAD_TOKENS = 12
TOOL_BLOCK_OVERHEAD_TOKENS = 24
FIXED_BINARY_BLOCK_TOKENS = 2_000

MODEL_CONTEXT_WINDOWS = {
    "claude-opus-4-20250514": 200_000,
    "claude-sonnet-4-20250514": 200_000,
    "claude-haiku-3-20250307": 200_000,
    "claude-3-5-sonnet-20241022": 200_000,
    "claude-3-5-haiku-20241022": 200_000,
    "claude-3-opus-20240229": 200_000,
}

def rough_token_count(content: str, chars_per_token: int = TEXT_CHARS_PER_TOKEN) -> int:
    return max(1, round(len(content) / chars_per_token))

def estimate_content_block_tokens(content) -> int:
    """Estimate tokens for a single content block or block list."""
    if isinstance(content, str):
        return rough_token_count(content)
    if not isinstance(content, list):
        return 0
    total = 0
    for block in content:
        btype = getattr(block, "type", None) or (block.get("type") if isinstance(block, dict) else None)
        if btype == "text":
            text = getattr(block, "text", None) or (block.get("text", "") if isinstance(block, dict) else "")
            total += rough_token_count(text)
        elif btype == "tool_use":
            name = getattr(block, "name", None) or (block.get("name", "") if isinstance(block, dict) else "")
            inp = getattr(block, "input", None) or (block.get("input", {}) if isinstance(block, dict) else {})
            total += TOOL_BLOCK_OVERHEAD_TOKENS
            total += rough_token_count(name)
            total += rough_token_count(json.dumps(inp), JSON_CHARS_PER_TOKEN)
        elif btype == "tool_result":
            c = getattr(block, "content", None) or (block.get("content", "") if isinstance(block, dict) else "")
            serialized = c if isinstance(c, str) else json.dumps(c)
            total += TOOL_BLOCK_OVERHEAD_TOKENS + rough_token_count(serialized, JSON_CHARS_PER_TOKEN)
        elif btype in ("image", "document"):
            total += FIXED_BINARY_BLOCK_TOKENS
        else:
            total += rough_token_count(json.dumps(block) if not isinstance(block, str) else block, JSON_CHARS_PER_TOKEN)
    return total

def estimate_message_tokens(message) -> int:
    content = message.get("content", "") if isinstance(message, dict) else getattr(message, "content", "")
    return MESSAGE_OVERHEAD_TOKENS + estimate_content_block_tokens(content)

def rough_token_count_for_messages(messages: list) -> int:
    """Estimate total tokens with 4/3 inflation factor for heuristic underestimation."""
    raw = sum(estimate_message_tokens(m) for m in messages)
    return math.ceil((raw * 4) / 3)

def get_context_window_for_model(model: str) -> int:
    env_override = os.environ.get("CLAUDE_CODE_MAX_CONTEXT_TOKENS")
    if env_override:
        parsed = int(env_override)
        if parsed > 0:
            return parsed
    if model in MODEL_CONTEXT_WINDOWS:
        return MODEL_CONTEXT_WINDOWS[model]
    for key, value in MODEL_CONTEXT_WINDOWS.items():
        if key in model or model in key:
            return value
    return MODEL_CONTEXT_WINDOW_DEFAULT

def get_effective_context_window(model: str) -> int:
    window = get_context_window_for_model(model)
    reserved = min(MAX_OUTPUT_TOKENS_FOR_SUMMARY, window // 5)
    return window - reserved

def scale_buffer(buffer: int, effective_window: int) -> int:
    reference = 180_000
    if effective_window >= reference:
        return buffer
    return round(buffer * (effective_window / reference))

def build_token_budget_snapshot(messages, model="claude-sonnet-4-20250514", system_prompt=None, usage=None, usage_anchor_index=None) -> dict:
    """Build a complete budget snapshot for a conversation."""
    system_tokens = rough_token_count(system_prompt) + MESSAGE_OVERHEAD_TOKENS if system_prompt else 0
    if usage and usage_anchor_index is not None and usage_anchor_index >= 0:
        suffix = messages[usage_anchor_index + 1:]
        estimated = (usage.get("input_tokens", 0) + usage.get("output_tokens", 0)
                     + usage.get("cache_creation_input_tokens", 0) + usage.get("cache_read_input_tokens", 0)
                     + rough_token_count_for_messages(suffix) + system_tokens)
    else:
        estimated = rough_token_count_for_messages(messages) + system_tokens

    window = get_context_window_for_model(model)
    effective = get_effective_context_window(model)
    return {
        "estimated_conversation_tokens": estimated,
        "context_window": window,
        "effective_context_window": effective,
        "auto_compact_threshold": max(0, effective - scale_buffer(AUTOCOMPACT_BUFFER_TOKENS, effective)),
        "manual_compact_threshold": max(0, effective - scale_buffer(MANUAL_COMPACT_BUFFER_TOKENS, effective)),
    }

# ── Demo ──
messages = [
    {"role": "user", "content": "What is the capital of France?"},
    {"role": "assistant", "content": [{"type": "text", "text": "The capital of France is Paris."}]},
    {"role": "user", "content": "Tell me more about its history."},
]

budget = build_token_budget_snapshot(messages)
print("=== Token Budget Snapshot ===")
for k, v in budget.items():
    print(f"  {k}: {v:,}")

print(f"\nContext window: {budget['context_window']:,} tokens")
print(f"Effective window: {budget['effective_context_window']:,} tokens")
print(f"Auto-compact at: {budget['auto_compact_threshold']:,} tokens")

### 4.1 Hybrid Usage + Estimation

When the API returns actual usage data, the system **anchors** on that and only estimates
for messages added after the anchor point.

In [ ]:
def token_count_with_estimation(messages, usage=None, usage_anchor_index=None, system_prompt=None) -> int:
    """
    Hybrid counting: API-reported usage as anchor + heuristic estimation for the suffix.
    This is more accurate than pure estimation for mid-conversation checks.
    """
    system_tokens = rough_token_count(system_prompt) + MESSAGE_OVERHEAD_TOKENS if system_prompt else 0
    if usage and usage_anchor_index is not None and usage_anchor_index >= 0:
        anchor_total = (usage.get("input_tokens", 0) + usage.get("output_tokens", 0)
                        + usage.get("cache_creation_input_tokens", 0) + usage.get("cache_read_input_tokens", 0))
        suffix = messages[usage_anchor_index + 1:]
        return anchor_total + rough_token_count_for_messages(suffix) + system_tokens
    return rough_token_count_for_messages(messages) + system_tokens

# Demo: anchor on turn 1, estimate turn 2
api_usage = {"input_tokens": 500, "output_tokens": 200, "cache_creation_input_tokens": 0, "cache_read_input_tokens": 0}
all_msgs = [
    {"role": "user", "content": "Hello"},
    {"role": "assistant", "content": "Hi there!"},
    {"role": "user", "content": "What is 2+2?"},
]

pure_estimate = token_count_with_estimation(all_msgs)
hybrid = token_count_with_estimation(all_msgs, usage=api_usage, usage_anchor_index=1)
print(f"Pure estimation: {pure_estimate} tokens")
print(f"Hybrid (anchor on msg[1]): {hybrid} tokens")
print(f"API-reported anchor: {sum(api_usage.values())} tokens")

---
## 5. MCP Name Utilities (`src/services/mcp/normalization.ts` + `mcpStringUtils.ts`)

The Anthropic API requires tool names matching `^[a-zA-Z0-9_-]{1,64}$`.
MCP names allow much broader character sets, so normalization replaces invalid chars with `_`.

Convention: `mcp__<normalizedServer>__<normalizedTool>`

In [ ]:
def normalize_name_for_mcp(name: str) -> str:
    """Replace any non-alphanumeric/non-dash/non-underscore character with _."""
    return re.sub(r"[^a-zA-Z0-9_-]", "_", name)

def build_mcp_tool_name(server_name: str, tool_name: str) -> str:
    return f"mcp__{normalize_name_for_mcp(server_name)}__{normalize_name_for_mcp(tool_name)}"

def is_mcp_tool_name(name: str) -> bool:
    return name.startswith("mcp__")

def parse_mcp_tool_name(full_name: str) -> Optional[dict]:
    parts = full_name.split("__")
    if len(parts) < 3 or parts[0] != "mcp" or not parts[1]:
        return None
    return {"server_name": parts[1], "tool_name": "__".join(parts[2:])}

# Demo
print("=== MCP Name Utilities ===")
print(f"normalize('my.server name') = {normalize_name_for_mcp('my.server name')}")
print(f"build('github', 'search_repos') = {build_mcp_tool_name('github', 'search_repos')}")
print(f"build('my server', 'do.thing') = {build_mcp_tool_name('my server', 'do.thing')}")
print(f"isMcp('mcp__github__search') = {is_mcp_tool_name('mcp__github__search')}")
print(f"isMcp('Read') = {is_mcp_tool_name('Read')}")
print(f"parse('mcp__github__search_repos') = {parse_mcp_tool_name('mcp__github__search_repos')}")

---
## 6. MCP Registry (`src/services/mcp/registry.ts`)

In-memory store of MCP server connections + their adapted tools. The `/mcp` command
reads this to render its status panel.

In [ ]:
class McpRegistry:
    """In-memory registry of MCP server connections and their tools."""

    def __init__(self):
        self._entries: dict[str, dict] = {}

    def set_entry(self, name: str, connection: McpServerConnection, tools: list):
        self._entries[name] = {"connection": connection, "tools": tools}

    def delete_entry(self, name: str):
        self._entries.pop(name, None)

    def get_all(self) -> list:
        return list(self._entries.values())

    def get_entry(self, name: str) -> Optional[dict]:
        return self._entries.get(name)

    def clear(self):
        self._entries.clear()

    def total_tools(self) -> int:
        return sum(len(e["tools"]) for e in self._entries.values())

# Demo
registry = McpRegistry()
registry.set_entry("github", ConnectedMcpServer(name="github", type="connected"), ["search", "create_pr"])
registry.set_entry("filesystem", PendingMcpServer(name="filesystem", type="pending"), [])
print(f"Registry entries: {len(registry.get_all())}")
print(f"Total tools: {registry.total_tools()}")
print(f"GitHub entry: type={registry.get_entry('github')['connection'].type}")

---
## 7. MCP Configuration Loading (`src/services/mcp/config.ts`)

Loads MCP server definitions from two JSON files:
| Source | Path | Priority |
|--------|------|----------|
| User | `~/.easy-agent/settings.json` | Lower |
| Project | `<cwd>/.easy-agent/settings.json` | Higher |

Project overrides user on name conflicts.

In [ ]:
def validate_server_config(name: str, raw: dict, scope: str) -> tuple:
    """
    Validate a single server config. Returns (ok, value_or_error).
    Supports stdio (default), http, and sse transports.
    """
    if not raw or not isinstance(raw, dict):
        return (False, f"mcpServers.{name} must be an object")

    transport_type = raw.get("type")
    if transport_type not in (None, "stdio", "http", "sse"):
        return (False, f"unsupported transport '{transport_type}'")

    if transport_type in ("http", "sse"):
        return _validate_remote_config(name, raw, scope, transport_type)
    return _validate_stdio_config(name, raw, scope)

def _validate_stdio_config(name, obj, scope):
    if not isinstance(obj.get("command"), str) or not obj["command"].strip():
        return (False, f"mcpServers.{name} ({scope}): 'command' required")
    if "args" in obj and not isinstance(obj["args"], list):
        return (False, f"mcpServers.{name} ({scope}): 'args' must be array")
    validated = {"type": "stdio", "command": obj["command"], "args": obj.get("args", []), "env": obj.get("env", {})}
    return (True, validated)

def _validate_remote_config(name, obj, scope, transport_type):
    url = obj.get("url")
    if not isinstance(url, str) or not url.strip():
        return (False, f"mcpServers.{name} ({scope}): '{transport_type}' requires 'url'")
    try:
        from urllib.parse import urlparse
        urlparse(url)
    except Exception:
        return (False, f"mcpServers.{name} ({scope}): invalid URL: {url}")
    validated = {"type": transport_type, "url": url, "headers": obj.get("headers", {})}
    return (True, validated)

def load_mcp_configs_simulated() -> dict:
    """
    Simulate loading from two settings files. In the real codebase,
    this reads ~/.easy-agent/settings.json and <cwd>/.easy-agent/settings.json.
    """
    user_settings = {
        "mcpServers": {
            "github": {"type": "stdio", "command": "npx", "args": ["-y", "@mcp/github"]},
            "web-search": {"type": "http", "url": "https://mcp.example.com/search"},
        }
    }
    project_settings = {
        "mcpServers": {
            "github": {"type": "stdio", "command": "npx", "args": ["-y", "@mcp/github@latest"]},
            "project-tools": {"type": "stdio", "command": "python", "args": ["-m", "mcp_server"]},
        }
    }

    errors = []
    servers = {}

    for scope, settings, path in [("user", user_settings, "~/.easy-agent/settings.json"),
                                   ("project", project_settings, "<cwd>/.easy-agent/settings.json")]:
        raw_servers = settings.get("mcpServers", {})
        for srv_name, srv_config in raw_servers.items():
            ok, result = validate_server_config(srv_name, srv_config, scope)
            if ok:
                result["scope"] = scope
                servers[srv_name] = result
            else:
                errors.append(result)

    return {"servers": servers, "errors": errors}

# Demo — project overrides user for 'github'
config = load_mcp_configs_simulated()
print("=== MCP Config Loading ===")
print(f"Loaded servers: {list(config['servers'].keys())}")
print(f"Errors: {config['errors']}")
print(f"\ngithub (overridden by project):")
print(f"  command: {config['servers']['github']['command']}")
print(f"  args: {config['servers']['github']['args']}")
print(f"  scope: {config['servers']['github']['scope']}")
print(f"\nweb-search (from user):")
print(f"  type: {config['servers']['web-search']['type']}")
print(f"  url: {config['servers']['web-search']['url']}")

---
## 8. MCP Client Connection Management (`src/services/mcp/client.ts`)

The most complex file in the MCP subsystem (~445 lines). Manages transport creation,
handshake with 30s timeout, connection caching, and graceful cleanup.

**Key patterns:**
- **TransportBundle** — encapsulates transport + description + stderr + cleanup per type
- **Cache key** includes full transport config so settings.json edits auto-invalidate
- **Signal escalation** for stdio: SIGINT (100ms) → SIGTERM (400ms) → SIGKILL

In [ ]:
CONNECT_TIMEOUT_MS = 30_000

class McpClient:
    """
    Python port of the MCP client connection manager.
    Demonstrates the transport factory, caching, and cleanup patterns.
    """

    def __init__(self):
        self._connection_cache: dict[str, Any] = {}  # key → connection result
        self._active_connections: dict[str, ConnectedMcpServer] = {}

    def get_cache_key(self, name: str, config: dict) -> str:
        """Build stable cache key from server name + transport-specific config."""
        if config.get("type") in ("http", "sse"):
            return f"{name}:{json.dumps({'type': config['type'], 'url': config['url'], 'headers': config.get('headers', {})}, sort_keys=True)}"
        return f"{name}:{json.dumps({'type': 'stdio', 'command': config.get('command', ''), 'args': config.get('args', []), 'env': config.get('env', {})}, sort_keys=True)}"

    def connect_simulated(self, name: str, config: dict) -> McpServerConnection:
        """
        Simulate the connectToServer() flow:
        1. Check cache
        2. Create transport (stdio/http/sse)
        3. Handshake with timeout
        4. Return ConnectedMcpServer or FailedMcpServer
        """
        key = self.get_cache_key(name, config)
        if key in self._connection_cache:
            print(f"  [{name}] Cache hit — reusing connection")
            return self._connection_cache[key]

        transport_type = config.get("type", "stdio")
        print(f"  [{name}] Connecting via {transport_type}...")

        if transport_type == "stdio":
            print(f"  [{name}] Spawning: {config['command']} {' '.join(config.get('args', []))}")
        elif transport_type in ("http", "sse"):
            print(f"  [{name}] Connecting to {config['url']}")

        # Simulate successful connection
        connection = ConnectedMcpServer(
            name=name,
            type="connected",
            config=config,
            server_info={"name": f"{name}-server", "version": "1.0.0"},
        )
        self._connection_cache[key] = connection
        self._active_connections[name] = connection
        return connection

    def escalated_kill(self, name: str, pid: int):
        """
        Stdio cleanup escalation: SIGINT (100ms) → SIGTERM (400ms) → SIGKILL.
        Total cap ~500ms so CLI exit isn't held up.
        """
        signals = ["SIGINT", "SIGTERM", "SIGKILL"]
        delays = [0.1, 0.4, 0]
        for sig, delay in zip(signals, delays):
            print(f"  [{name}] Sending {sig} to PID {pid}")
            # In real code: process.kill(pid, sig)
            time.sleep(delay)
            # In real code: check if process still alive
            print(f"  [{name}] Process exited after {sig}")

    def cleanup_all(self):
        """Process-level cleanup — called on SIGINT/SIGTERM/beforeExit."""
        conns = list(self._active_connections.values())
        self._active_connections.clear()
        print(f"Cleaning up {len(conns)} MCP connections...")
        for conn in conns:
            print(f"  [{conn.name}] Cleanup complete")

# Demo
client = McpClient()
configs = {
    "github": {"type": "stdio", "command": "npx", "args": ["-y", "@mcp/github"]},
    "web-search": {"type": "http", "url": "https://mcp.example.com/search"},
}
print("=== MCP Client Connection ===")
for name, cfg in configs.items():
    conn = client.connect_simulated(name, cfg)
    print(f"  Result: {conn.name} → {conn.type}")

# Cache hit demo
print("\n--- Second call (cache hit) ---")
client.connect_simulated("github", configs["github"])

print("\n--- Process cleanup ---")
client.cleanup_all()

---
## 9. MCP Tool Discovery & Adaptation (`src/services/mcp/fetchTools.ts`)

Bridges the MCP protocol's tool model and Easy Agent's local `Tool` interface.
Each MCP tool gets wrapped in an adapter that:
- Uses the **prefixed name** (`mcp__server__tool`) for the local registry
- Sends the **original name** to the MCP server on `call()`

In [ ]:
MAX_MCP_DESCRIPTION_LENGTH = 2048

def stringify_mcp_content(content: list) -> str:
    """Map MCP CallToolResult.content[] blocks to a single string."""
    if not isinstance(content, list):
        return ""
    parts = []
    for block in content:
        btype = block.get("type", "unknown") if isinstance(block, dict) else getattr(block, "type", "unknown")
        if btype == "text":
            parts.append(block.get("text", "") if isinstance(block, dict) else getattr(block, "text", ""))
        elif btype == "image":
            mime = block.get("mimeType", "?") if isinstance(block, dict) else getattr(block, "mimeType", "?")
            data = block.get("data", "") if isinstance(block, dict) else getattr(block, "data", "")
            parts.append(f"[image: {mime}, {len(data)} base64 chars]")
        elif btype == "resource":
            r = block.get("resource", {}) if isinstance(block, dict) else getattr(block, "resource", {})
            parts.append(r.get("text", f"[resource: {r.get('uri', '<no uri>')}]" if isinstance(r, dict) else str(r)))
        else:
            parts.append(f"[{btype} block]")
    return "\n".join(parts)

class McpToolAdapter:
    """Wraps an MCP tool into the local Tool interface."""

    def __init__(self, connection_name: str, mcp_tool_name: str, description: str,
                 input_schema: dict, is_read_only: bool, connection_client=None):
        self.name = build_mcp_tool_name(connection_name, mcp_tool_name)
        self.original_name = mcp_tool_name
        self.description = (description or "")[:MAX_MCP_DESCRIPTION_LENGTH]
        self.input_schema = input_schema
        self._is_read_only = is_read_only

    def is_read_only(self) -> bool:
        return self._is_read_only

    def is_enabled(self) -> bool:
        return True

    def call(self, raw_input: dict) -> dict:
        """
        Forward to the MCP server via client.request({method: 'tools/call'}).
        In the real code, this is async and uses the MCP SDK client.
        """
        # Simulated response
        return {"content": f"Result from {self.original_name}({json.dumps(raw_input)})", "is_error": False}

def build_tool_adapter(connection_name: str, mcp_tool: dict) -> McpToolAdapter:
    """Build a local Tool adapter from a single MCP tool descriptor."""
    return McpToolAdapter(
        connection_name=connection_name,
        mcp_tool_name=mcp_tool["name"],
        description=mcp_tool.get("description", ""),
        input_schema=mcp_tool.get("inputSchema", {"type": "object", "properties": {}}),
        is_read_only=mcp_tool.get("annotations", {}).get("readOnlyHint", False),
    )

# Simulate tools/list response
mcp_tools_response = [
    {"name": "search_repos", "description": "Search GitHub repositories", "inputSchema": {"type": "object", "properties": {"query": {"type": "string"}}}},
    {"name": "create_pr", "description": "Create a pull request", "inputSchema": {"type": "object", "properties": {"title": {"type": "string"}, "body": {"type": "string"}}}},
]

print("=== MCP Tool Discovery ===")
adapters = []
for mcp_tool in mcp_tools_response:
    adapter = build_tool_adapter("github", mcp_tool)
    adapters.append(adapter)
    print(f"  {adapter.name} — {adapter.description[:50]}")
    print(f"    original_name={adapter.original_name}, is_read_only={adapter.is_read_only()}")

print(f"\n--- Tool Call ---")
result = adapters[0].call({"query": "easy-agent"})
print(f"  call(search_repos, {{query: 'easy-agent'}}) → {result}")

---
## 10. Skills Frontmatter Parsing (`src/services/skills/parseFrontmatter.ts`)

Splits a `---\n...\n---\n<body>` document into YAML frontmatter + markdown body.

In [ ]:
FRONTMATTER_RE = re.compile(r'^---\r?\n([\s\S]*?)\r?\n---\r?\n?([\s\S]*)$', re.MULTILINE)

def split_frontmatter(content: str) -> dict:
    """Split + parse a SKILL.md document. Never throws."""
    match = FRONTMATTER_RE.match(content)
    if not match:
        return {"raw": {}, "body": content}

    yaml_text, body = match.group(1), match.group(2)
    try:
        # Simulate YAML parse (real code uses `yaml` package)
        parsed = {}
        for line in yaml_text.strip().split("\n"):
            if ":" in line:
                key, _, value = line.partition(":")
                parsed[key.strip()] = value.strip()
        if parsed and isinstance(parsed, dict):
            return {"raw": parsed, "body": body}
        return {"raw": {}, "body": body, "parse_error": "Frontmatter must be a YAML mapping"}
    except Exception as e:
        return {"raw": {}, "body": body, "parse_error": str(e)}

def as_string(value) -> Optional[str]:
    if isinstance(value, str):
        trimmed = value.strip()
        return trimmed if trimmed else None
    if isinstance(value, (int, float, bool)):
        return str(value)
    return None

def as_string_array(value) -> list:
    if isinstance(value, list):
        return [s.strip() for s in value if isinstance(s, str) and s.strip()]
    if isinstance(value, str):
        return [s.strip() for s in value.split(",") if s.strip()]
    return []

def as_boolean(value) -> bool:
    if isinstance(value, bool):
        return value
    if isinstance(value, str):
        return value.strip().lower() in ("true", "yes", "1")
    return False

def extract_fallback_description(body: str) -> str:
    """Extract first non-empty, non-heading paragraph from markdown body."""
    lines = body.split("\n")
    buf = []
    for raw_line in lines:
        line = raw_line.strip()
        if not line:
            if buf:
                break
            continue
        if not buf and line.startswith("#"):
            continue
        buf.append(line)
    return " ".join(buf).strip()

def normalize_frontmatter(raw: dict, body: str) -> SkillFrontmatter:
    """Normalize raw YAML into a SkillFrontmatter."""
    allowed_tools = as_string_array(raw.get("allowed-tools") or raw.get("allowedTools"))
    paths = as_string_array(raw.get("paths"))
    return SkillFrontmatter(
        name=as_string(raw.get("name")),
        description=as_string(raw.get("description")),
        when_to_use=as_string(raw.get("when_to_use") or raw.get("whenToUse")),
        allowed_tools=allowed_tools,
        argument_hint=as_string(raw.get("argument-hint") or raw.get("argumentHint")),
        disable_model_invocation=as_boolean(raw.get("disable-model-invocation") or raw.get("disableModelInvocation")),
        paths=paths if paths else None,
        has_fork_context=as_string(raw.get("context")) == "fork",
        raw=raw,
    )

# Demo
skill_md = '''---
name: test-reviewer
description: Reviews test files for coverage gaps
when_to_use: When the user asks to review tests
allowed-tools: Read, Grep, Glob
paths: "**/*.test.ts, **/*.spec.ts"
---

## Test Reviewer

This skill analyzes test files for missing edge cases
and coverage gaps.
'''

split = split_frontmatter(skill_md)
print("=== Frontmatter Parsing ===")
print(f"Raw frontmatter: {split['raw']}")
print(f"Body preview: {split['body'][:60].strip()!r}...")

fm = normalize_frontmatter(split["raw"], split["body"])
print(f"\nNormalized SkillFrontmatter:")
print(f"  name: {fm.name}")
print(f"  description: {fm.description}")
print(f"  when_to_use: {fm.when_to_use}")
print(f"  allowed_tools: {fm.allowed_tools}")
print(f"  paths: {fm.paths}")
print(f"  disable_model_invocation: {fm.disable_model_invocation}")

# Fallback description demo
no_desc_md = '''---
name: my-skill
---

# My Skill

This skill does amazing things with code analysis.
It checks for bugs and style issues.
'''
split2 = split_frontmatter(no_desc_md)
fallback = extract_fallback_description(split2["body"])
print(f"\nFallback description (no frontmatter desc): {fallback!r}")

---
## 11. Skills Registry — Dual-Map Store (`src/services/skills/registry.ts`)

Two maps implement lazy activation:
| Map | Contains | Visible To |
|-----|----------|-----------|
| `dynamic` | Skills without `paths`, plus activated conditional | Model + user |
| `conditional` | Skills with `paths` that haven't matched yet | User only |

Activation is **one-way and sticky** — once promoted, a skill stays visible.

In [ ]:
class SkillsRegistry:
    def __init__(self):
        self._dynamic: dict[str, Skill] = {}
        self._conditional: dict[str, Skill] = {}
        self._initialized = False

    def set_skills(self, skills: list):
        """Replace registry. Split by presence of frontmatter.paths."""
        self._dynamic.clear()
        self._conditional.clear()
        for skill in skills:
            if skill.frontmatter and skill.frontmatter.paths:
                self._conditional[skill.name] = skill
            else:
                self._dynamic[skill.name] = skill
        self._initialized = True

    def get_model_visible_skills(self) -> list:
        """Skills visible to the model (excludes disable_model_invocation)."""
        return [s for s in self._dynamic.values()
                if not (s.frontmatter and s.frontmatter.disable_model_invocation)]

    def get_all_user_invocable_skills(self) -> list:
        """All skills the user can invoke via /name (both maps)."""
        return list(self._dynamic.values()) + list(self._conditional.values())

    def find_skill(self, name: str) -> Optional[Skill]:
        return self._dynamic.get(name) or self._conditional.get(name)

    def activate_conditional(self, name: str) -> bool:
        """Promote conditional → dynamic. Returns True if skill was previously latent."""
        skill = self._conditional.pop(name, None)
        if not skill:
            return False
        self._dynamic[name] = skill
        return True

    def list_conditional(self) -> list:
        return list(self._conditional.values())

# Demo
reg = SkillsRegistry()
skills = [
    Skill(name="code-review", description="Reviews code for bugs",
          frontmatter=SkillFrontmatter(description="Reviews code")),
    Skill(name="test-reviewer", description="Reviews test files",
          frontmatter=SkillFrontmatter(description="Reviews tests", paths=["**/*.test.ts"])),
    Skill(name="deploy-helper", description="Helps with deployments",
          frontmatter=SkillFrontmatter(description="Deploy help")),
]
reg.set_skills(skills)

print("=== Skills Registry ===")
print(f"Model-visible: {[s.name for s in reg.get_model_visible_skills()]}")
print(f"Conditional: {[s.name for s in reg.list_conditional()]}")
print(f"All user-invocable: {[s.name for s in reg.get_all_user_invocable_skills()]}")

# Activate the conditional skill
activated = reg.activate_conditional("test-reviewer")
print(f"\nActivated 'test-reviewer': {activated}")
print(f"Model-visible now: {[s.name for s in reg.get_model_visible_skills()]}")
print(f"Conditional now: {[s.name for s in reg.list_conditional()]}")

### 11.1 Conditional Activation (`src/services/skills/conditional.ts`)

When the agent touches a file (Read/Write/Edit/Glob), the system checks if any
conditional skill's `paths` patterns match. Uses gitignore-style matching.

In [ ]:
def activate_conditional_skills_for_paths(file_paths: list, cwd: str, registry: SkillsRegistry) -> list:
    """
    Try to activate every still-conditional skill against the given file paths.
    Returns names of skills that just became visible.
    """
    candidates = registry.list_conditional()
    if not candidates or not file_paths:
        return []

    # Convert to repo-relative paths
    relative_paths = []
    for p in file_paths:
        abs_path = os.path.abspath(p) if not os.path.isabs(p) else p
        rel = os.path.relpath(abs_path, cwd)
        if rel and not rel.startswith("..") and not os.path.isabs(rel):
            relative_paths.append(rel.replace(os.sep, "/"))

    if not relative_paths:
        return []

    activated = []
    for skill in candidates:
        patterns = skill.frontmatter.paths if skill.frontmatter else None
        if not patterns:
            continue
        # Simple glob-style matching (real code uses `ignore` package)
        for pattern in patterns:
            for rp in relative_paths:
                # Simplified: check if file ends with the pattern suffix
                clean_pattern = pattern.replace("**/", "").replace("*", "")
                if clean_pattern and clean_pattern in rp:
                    if registry.activate_conditional(skill.name):
                        activated.append(skill.name)
                    break
            if skill.name in activated:
                break

    return activated

def extract_tool_file_paths(tool_name: str, tool_input: dict) -> list:
    """Extract file-path-shaped fields from tool inputs."""
    if tool_name in ("Read", "Write", "Edit"):
        fp = tool_input.get("file_path")
        return [fp] if isinstance(fp, str) else []
    if tool_name == "Glob":
        root = tool_input.get("path")
        return [root] if isinstance(root, str) else []
    return []

# Demo
reg = SkillsRegistry()
reg.set_skills([
    Skill(name="code-review", description="Reviews code",
          frontmatter=SkillFrontmatter(description="Reviews code")),
    Skill(name="test-reviewer", description="Reviews tests",
          frontmatter=SkillFrontmatter(description="Reviews tests", paths=["**/*.test.ts"])),
])

print("=== Conditional Activation ===")
print(f"Before: conditional = {[s.name for s in reg.list_conditional()]}")

# Agent reads a test file
paths = extract_tool_file_paths("Read", {"file_path": "/repo/src/auth.test.ts"})
print(f"Extracted paths from Read: {paths}")

activated = activate_conditional_skills_for_paths(paths, "/repo", reg)
print(f"Activated: {activated}")
print(f"After: conditional = {[s.name for s in reg.list_conditional()]}")
print(f"Model-visible: {[s.name for s in reg.get_model_visible_skills()]}")

---
## 12. Skills Budget & System Prompt Injection (`src/services/skills/budget.ts`)

Three-tier degradation to fit skill listings within a character budget (default 8,000 chars ≈ 2,000 tokens):

| Tier | Format | When |
|------|--------|------|
| 1 | `- name: full description (≤250 chars)` | Fits within budget |
| 2 | `- name: shrunk description (shared budget)` | Tier 1 too large, but ≥20 chars/skill fits |
| 3 | `- name` | Everything else |

In [ ]:
MAX_LISTING_DESC_CHARS = 250
MIN_DESC_CHARS_PER_SKILL = 20
DEFAULT_BUDGET_CHARS = 8000

def truncate_desc(desc: str, max_len: int) -> str:
    if len(desc) <= max_len:
        return desc
    if max_len <= 1:
        return "…"
    return desc[:max_len - 1].rstrip() + "…"

def build_line(skill: Skill, desc_max: int) -> str:
    capped = min(desc_max, MAX_LISTING_DESC_CHARS)
    full_desc = f"{skill.description} — {skill.when_to_use}" if skill.when_to_use else skill.description
    return f"- {skill.name}: {truncate_desc(full_desc, capped)}"

def format_skills_within_budget(skills: list, budget: int = DEFAULT_BUDGET_CHARS) -> str:
    """
    Tier 1: full descriptions (capped at 250 chars each).
    Tier 2: shrink descriptions equally to fit (≥ 20 chars each).
    Tier 3: names only.
    """
    if not skills:
        return ""

    # Tier 1
    tier1 = [build_line(s, MAX_LISTING_DESC_CHARS) for s in skills]
    tier1_total = sum(len(line) + 1 for line in tier1)
    if tier1_total <= budget:
        return "\n".join(tier1)

    # Tier 2
    prefix_cost = sum(len(f"- {s.name}: ") + 1 for s in skills)
    desc_budget = budget - prefix_cost
    if desc_budget >= len(skills) * MIN_DESC_CHARS_PER_SKILL:
        per_desc = max(MIN_DESC_CHARS_PER_SKILL, desc_budget // len(skills))
        tier2 = [build_line(s, per_desc) for s in skills]
        if sum(len(line) + 1 for line in tier2) <= budget:
            return "\n".join(tier2)

    # Tier 3
    return "\n".join(f"- {s.name}" for s in skills)

def format_skills_system_reminder(skills: list) -> str:
    """Build the <system-reminder> block for the system prompt."""
    if not skills:
        return ""
    listing = format_skills_within_budget(skills)
    if not listing:
        return ""
    return "\n".join([
        "<system-reminder>",
        "Available skills you can invoke via the `Skill` tool. Each line is `- <name>: <description>`.",
        'Call `Skill(skill="<name>", args="<optional args>")` when the request matches one of these.',
        "",
        listing,
        "</system-reminder>",
    ])

# Demo
skills = [
    Skill(name="code-review", description="Analyzes code for bugs, style issues, and security vulnerabilities",
          when_to_use="When reviewing PRs or asking for code feedback"),
    Skill(name="test-generator", description="Generates comprehensive test suites"),
    Skill(name="refactor-helper", description="Suggests refactoring improvements for better maintainability"),
    Skill(name="doc-writer", description="Generates documentation from code comments and structure"),
]

print("=== Skills Budget — Tier 1 (full) ===")
t1 = format_skills_within_budget(skills)
print(t1)

print(f"\n=== Tier 2 (shrunk, budget=200) ===")
t2 = format_skills_within_budget(skills, budget=200)
print(t2)

print(f"\n=== Tier 3 (names only, budget=50) ===")
t3 = format_skills_within_budget(skills, budget=50)
print(t3)

print(f"\n=== System Reminder ===")
reminder = format_skills_system_reminder(skills)
print(reminder)

---
## 13. Stream Debug Infrastructure (`src/utils/streamDebug.ts`)

Opt-in JSONL logging of every raw SSE event. Zero-cost when disabled (single boolean check).

In [ ]:
DEBUG_STREAM = os.environ.get("EASY_AGENT_DEBUG_STREAM") == "1"

_debug_log: list = []  # In-memory for notebook demo

def write_stream_debug(kind: str, payload):
    """Append a single JSON record to the debug log. No-op when disabled."""
    if not DEBUG_STREAM:
        return
    record = {"ts": datetime.now().isoformat(), "kind": kind, "payload": payload}
    _debug_log.append(record)
    # In real code: append to ~/.easy-agent/stream-debug.log

# Demo — enable debug mode temporarily
# Re-read the flag after setting the env var
os.environ["EASY_AGENT_DEBUG_STREAM"] = "1"
DEBUG_STREAM = True  # Update module-level flag

write_stream_debug("request", {"model": "claude-sonnet-4-20250514", "messageCount": 3})
write_stream_debug("event", {"type": "message_start", "message": {"id": "msg_01"}})
write_stream_debug("assembled", {"stopReason": "end_turn", "blockCount": 2})

print("=== Stream Debug Records ===")
for rec in _debug_log:
    print(f"  [{rec['kind']}] {json.dumps(rec['payload'])[:80]}")

# Reset
DEBUG_STREAM = False
print(f"\nDebug enabled: {DEBUG_STREAM}")


---
## Summary

| Cell | Component | Source File | Key Concept |
|------|-----------|-------------|-------------|
| 1 | ContentBlock, Message, Usage | `src/types/message.ts` | Type hierarchy for all LLM content |
| 2 | StreamEvent discriminated union | `src/types/message.ts` | 6 event kinds for incremental delivery |
| 3 | McpServerConnection states | `src/types/mcp.ts` | Pending → Connected/Failed lifecycle |
| 4 | Skill, SkillFrontmatter | `src/types/types.ts` | Declarative prompt templates |
| 5 | API Client singleton | `src/services/api/client.ts` | Lazy init, token limit constants |
| 6 | streamMessage() generator | `src/services/api/streaming.ts` | Core communication primitive |
| 7 | Per-index JSON accumulation | `src/services/api/streaming.ts` | Prevents overlap corruption |
| 8 | Escalated retry | `src/services/api/streaming.ts` | 8K → 64K on truncation |
| 9 | Token estimation | `src/utils/tokens.ts` | Character-based heuristic + 4/3 inflation |
| 10 | Hybrid usage+estimation | `src/utils/tokens.ts` | API anchor + estimate suffix |
| 11 | MCP name utilities | `src/services/mcp/normalization.ts`, `mcpStringUtils.ts` | `mcp__server__tool` convention |
| 12 | MCP registry | `src/services/mcp/registry.ts` | In-memory connection+tool store |
| 13 | MCP config loading | `src/services/mcp/config.ts` | Two-scope settings, validation |
| 14 | MCP client connection | `src/services/mcp/client.ts` | Transport factory, cache, cleanup |
| 15 | MCP tool discovery | `src/services/mcp/fetchTools.ts` | tools/list → Tool adapter |
| 16 | Skills frontmatter | `src/services/skills/parseFrontmatter.ts` | YAML split + normalization |
| 17 | Skills registry | `src/services/skills/registry.ts` | Dual-map, one-way activation |
| 18 | Conditional activation | `src/services/skills/conditional.ts` | Gitignore-style path matching |
| 19 | Skills budget | `src/services/skills/budget.ts` | Three-tier degradation |
| 20 | Stream debug | `src/utils/streamDebug.ts` | Opt-in JSONL logging |

> **Next step:** To see the real streaming in action, set `ANTHROPIC_AUTH_TOKEN` and
> replace `stream_message_simulated()` with the actual Anthropic SDK call.